# 015 — Costo uniforme, búsqueda voraz y A*

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("search", seed=15)
assert result["kind"] == "search"
assert result["evidence"]
show(result)


## Solución 1 — UCS

```text
extrae S (g=0) → encola A (g=1), B (g=4)
extrae A (g=1) → encola B (g=3) ← mejora la copia g=4, G (g=13)
extrae B (g=3) → encola G (g=6) ← mejora la copia g=13
extrae G (g=6) → objetivo ✔
```

Orden: `S(0), A(1), B(3), G(6)`. Camino `S→A→B→G`, costo **6** (óptimo).
Nótese cómo UCS *reabre* mejores copias de B y G: por eso la cola de prioridad
debe permitir actualizar (o insertar duplicados y descartar los peores).


## Solución 2 — Voraz

Desde `S`, compara `h(A)=4` con `h(B)=2` → elige `B`; desde `B`, `h(G)=0` → `G`.
Camino `S→B→G`, costo real `4+3 = 7` — **subóptimo**. La decisión que lo
estropea es la primera: ignora que llegar a `B` vía `A` cuesta 3 en lugar de 4,
porque la voraz no mira `g` (el costo ya pagado) en absoluto.


## Solución 3 — A*

```text
extrae S: f = 0+5 = 5   → A (f=1+4=5), B (f=4+2=6)
extrae A: f = 5         → B (g=3, f=3+2=5), G (g=13, f=13)
extrae B: f = 5         → G (g=6, f=6)
extrae G: f = 6         → objetivo ✔  camino S→A→B→G, costo 6
```

Admisibilidad: `h*(S)=6 ≥ 5 ✔`, `h*(A)=5 ≥ 4 ✔`, `h*(B)=3 ≥ 2 ✔`, `h*(G)=0 = 0 ✔`.
A* expande los mismos nodos que UCS aquí, pero con mejor ordenación: la copia
mala de G (f=13) nunca llega a extraerse.


In [ ]:
import heapq

grafo = {"S": [("A", 1), ("B", 4)], "A": [("B", 2), ("G", 12)], "B": [("G", 3)], "G": []}
h = {"S": 5, "A": 4, "B": 2, "G": 0}

def a_estrella(inicio, objetivo):
    frontera = [(h[inicio], 0, inicio, [inicio])]
    mejor_g = {}
    while frontera:
        f, g, nodo, camino = heapq.heappop(frontera)
        if nodo == objetivo:
            return camino, g
        if nodo in mejor_g and mejor_g[nodo] <= g:
            continue
        mejor_g[nodo] = g
        for hijo, costo in grafo[nodo]:
            heapq.heappush(frontera, (g + costo + h[hijo], g + costo, hijo, camino + [hijo]))
    return None, None

camino, costo = a_estrella("S", "G")
assert camino == ["S", "A", "B", "G"] and costo == 6
print("A* verificado ✔", camino, "costo", costo)


## Solución 4 — BFS vs. UCS

BFS y UCS coinciden exactamente cuando **todas las acciones cuestan lo mismo**:
entonces `g(n)` es proporcional a la profundidad y la cola de prioridad de UCS
se comporta como la FIFO de BFS. El grafo del laboratorio no declara pesos
(costo 1 por arista), así que su BFS ya es óptimo en costo; con pesos
heterogéneos, el mismo grafo exigiría UCS o A*.


In [ ]:
result = run_lab("search", seed=15)
assert result["result"]["cost"] == len(result["result"]["path"]) - 1
print("en costos unitarios, cost == número de aristas ✔")


## Reflexión

1. En el grafo de trabajo, la búsqueda voraz devuelve S→B→G (costo 7) y A* devuelve S→A→B→G (costo 6). ¿Qué información ignora la voraz que le cuesta la optimalidad?
2. ¿Por qué UCS y A* con h admisible garantizan optimalidad pero BFS solo la garantiza con costos uniformes? Da un contraejemplo concreto para BFS.
3. Si multiplicaras h por 3 (haciéndola inadmisible), A* seguiría encontrando *una* solución. ¿Qué garantía pierde exactamente y qué se gana a cambio (weighted A*)?
